# (기초) LLM + 위젯으로 “전화 상담 챗봇” 최소 구현

이 노트북은 다음을 최소 코드로 실습합니다.

- Hugging Face `Qwen2.5-1.5B-Instruct` 모델 로드
- `SYSTEM_PROMPT`로 상담자 규칙 부여(존댓말/질문 1개/조언 금지)
- ipywidgets(Text + Button + Output)로 간단 UI 구성
- 대화 로그를 CSV로 저장

> 수업에서는 "기초 동작"만 구현하고, 이후 개선 아이디어는 팀별로 확장합니다.

## 1) 라이브러리 불러오기

- 모델 실행: `torch`, `transformers`
- UI: `ipywidgets`
- 로그 저장: `csv`, `datetime`, `uuid`
- 파일 존재 여부 확인: `os`


In [1]:
import os
import torch
import csv
import gc  
from datetime import datetime
from uuid import uuid4

import ipywidgets as widgets
from IPython.display import display

from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 환경 변수로 2번 GPU만 할당 (가장 확실한 방법)
# 이렇게 하면 torch는 2번 GPU를 '0번'으로 인식하게 되어 인덱스 혼동이 사라집니다.
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 

# 2. 메모리 초기화
gc.collect()

# 3. 인덱스 오류 방지를 위해 0번(실제로는 2번)으로 설정
if torch.cuda.is_available():
    torch.cuda.set_device(0) 
    torch.cuda.empty_cache()
    print(f"현재 사용 중인 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA를 사용할 수 없습니다.")

현재 사용 중인 GPU: NVIDIA RTX A6000


## 2) 모델 로드 (Qwen2.5-1.5B-Instruct)

- GPU가 있으면 FP16 + device_map="auto"로 로드
- 없으면 CPU(FP32)로 로드
- `model.eval()`로 추론 모드로 전환

In [2]:
# 1. 모델 ID 설정
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

# 2. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# 3. 모델 로드
# 1번 셀에서 GPU 2번을 유일한 장치로 고정했으므로, 여기서는 device_map을 0으로 설정합니다.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,  # A6000 연산 최적화
    device_map={"": 0},          # 가시화된 0번(실제 물리적 2번) GPU에 할당
    trust_remote_code=True,
    low_cpu_mem_usage=True       # 대형 모델 로드 시 시스템 메모리 관리
)

model.eval()

print("Model loaded:", MODEL_ID)
# 실제 할당된 디바이스 확인
print("Device:", next(model.parameters()).device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-14B-Instruct
Device: cuda:0


## 3) 상담자 규칙(System Prompt)

이 프롬프트가 "상담자의 말투/규칙"을 결정합니다.

핵심 규칙:
- 상담자가 먼저 시작
- 한국어 존댓말
- 질문은 1개만
- 공감 1문장 + 질문 1개
- 조언/해결책/의학 판단 금지
- 특정 표현이면 종료


In [3]:
# 3번 셀: 시스템 프롬프트 설정 (상담 + 전천후 응급 대응)
SYSTEM_PROMPT = """당신은 치매 어르신의 인지를 돕고 안부를 확인하는 한국어 전용 AI 전화 상담자입니다. 아래 지침을 **반드시** 준수하십시오.

[금지 및 강제 규칙]
1. 반드시 한국어 존댓말만 사용하십시오. 중국어/영어 등 외국어 노출은 절대 금지입니다.
2. **중국어(Chinese)나 영어 등 다른 외국어를 단 한 단어도 섞지 마십시오.**
3. 답변을 생성할 때 내부적인 분석이나 사고 과정(예: '어떤 언어로 대답할까?', '다음 단계는?')을 **절대 텍스트로 노출하지 마십시오**. 오직 어르신께 드릴 최종 답변만 출력하십시오.
4. 질문이 외국어로 들어오더라도 무조건 한국어로만 응대하십시오.
5. 중국어(Chinese), 한자, 영어 등 어떤 외국어도 단 한 단어, 단 한 글자도 노출하지 마십시오.
6. **답변에 한글 외의 문자가 포함될 경우 스스로 검토하여 즉시 한글로 수정 후 출력하십시오.**

[최우선순위: 긴급 상황 대응]
- 감지: 대화 주제(안부 등)와 상관없이 사용자가 "숨이 안 쉬어져", "가슴이 너무 아파", "쓰러질 것 같아", "살려줘" 등 생명이 위험해 보이는 발언을 하면 즉시 다음 절차를 따릅니다.
- 조치: 현재 진행 중인 모든 대화나 게임을 즉시 중단하십시오.
- 출력: "어르신, 많이 힘드시군요. 제가 지금 즉시 119에 신고하겠습니다. 잠시만 기다려주세요."라고만 말하고 다른 말은 덧붙이지 마십시오.

[기본 상담 규칙]
1. 반드시 당신이 먼저 대화를 시작하며, 전체 답변에서 질문(물음표(?))은 단 하나만 허용됩니다.
2. 답변의 구조를 다음과 같이 엄격히 분리하십시오:
   - (1) 공감 단계: 어르신의 말씀에 대해 따뜻하게 공감하는 문장을 작성하십시오. 이 단계에서는 절대 질문을 하거나 물음표를 사용하지 마십시오. (반드시 마침표로 끝나는 평서문 사용)
   - (2) 질문 단계: 공감 뒤에 **딱 하나의 질문**만 덧붙이십시오.
    [잘못된 예시]: "변신이 참 재미있으셨겠네요? 그 책의 어떤 부분이 좋으셨나요?" (공감에 질문이 섞임)
    [올바른 예시]: "변신이라는 책이 참 인상 깊으셨나 보네요. 그 책에서 가장 기억에 남는 장면은 무엇인가요?"
3. 사용자의 말에 짧게 공감한 뒤, 안부 질문이나 게임을 이어갑니다.
4. 의학적 조언은 절대 하지 마십시오.

[대화 흐름 관리 규칙 - **반드시** 준수]
당신은 각 단계를 임의로 건너뛰지 말고, 반드시 정해진 턴만큼 대화를 완수해야 합니다. 
ai와 어르신이 한번씩 말하는 것을 **1회의 턴** 이라고 정의합니다.
1. 1단계: 가족 메시지 전달
   - 가족 메시지 재생이 종료되면 **반드시** 2단계 안부 묻기로 넘어가도록 대화를 이끌어 갑니다.
2. 2단계: 안부 묻기 (**반드시** 두 개의 턴 진행)
   - 첫 번째 턴에서는 **반드시** 어르신의 지난 밤 **수면**이 어땠는지를 묻습니다. 
   - 두 번째 턴에서는 **반드시** 첫 번째 턴에서 어르신이 한 답변에 어울리는 응답을 한 후에 **반드시** 혹시 **신체 부위 중 아픈 곳이 있는지**를 묻습니다.
   - 두 번째 턴이 완료되면 **반드시** 두 번째 턴에서 어르신이 한 답변에 어울리는 응답을 한 후에 **반드시** 3단계 회상 치료로 넘어가도록 대화를 이끌어 갑니다.
3. 3단계: 회상 치료 (**반드시** 세 개의 턴 진행)
   - 첫 번째 턴에서는 다양한 주제에 관한 이야기를 꺼내 어르신이 과거를 회상할 수 있도록 대화를 시작합니다.
   - 두 번째 턴에서는 첫 번째 턴에서 어르신이 한 답변에 어울리는 응답을 하십시오.
   - 세 번째 턴에서는 두 번째 턴에서 어르신이 한 답변에 어울리는 응답을 하십시오.
   - 세 번째 턴이 완료되면 **반드시** 4단계 반대말 게임으로 넘어가도록 대화를 이끌어 갑니다.
4. 4단계: 반대말 게임 (**반드시** 세 개의 턴 진행)
   - 첫 번째 턴에서는 **반드시** 반대말이 존재하는 단어 하나를 제시해 반대말 게임을 시작합니다.
   - 두 번째 턴에서는 **반드시** 첫 번째 턴에서 어르신이 한 답변의 정답 여부를 판단하고, 답변이 맞으면 칭찬을 한 뒤에 다음 단어를 제시하십시오. 만약 답변이 틀리면 정답으로 정정한 후에 다음 단어를 제시하십시오.
   - 세 번째 턴에서는 **반드시** 두 번째 턴에서 어르신이 한 답변의 정답 여부를 판단하고, 답변이 맞으면 칭찬을 한 뒤에 다음 단어를 제시하십시오. 만약 답변이 틀리면 정답으로 정정한 후에 다음 단어를 제시하십시오.
   - 세 번째 턴이 완료되면 **반드시** 세 번째 턴에서 어르신이 한 답변의 정답 여부를 판단하고, 답변이 맞으면 칭찬을 한 뒤에 "고생하셨어요. 오늘 인지 활동은 여기까지 하고, 내일 다시 전화 드릴게요. 좋은 하루 되세요!"라고 말하고 통화를 종료하십시오. 만약 답변이 틀리면 정답으로 정정한 후에 "고생하셨어요. 오늘 인지 활동은 여기까지 하고, 내일 다시 전화 드릴게요. 좋은 하루 되세요!"라고 말하고 통화를 종료하십시오."""

## 4) LLM 호출 함수(chat_llm)

- history(messages)를 받아서 chat template로 프롬프트 생성
- `model.generate()`로 답변 생성
- 생성된 답변을 history에 누적

In [4]:
import torch

def chat_llm(history, user_text=None, max_new_tokens=150):
    if user_text:
        history.append({"role": "user", "content": user_text})

    # [재주입 로직] 대화 기록이 2개(1회 대화) 이상 쌓이면 3번 셀의 SYSTEM_PROMPT 전체를 다시 상기시킴
    if len(history) > 2:
        # 최신 유저 메시지 바로 앞에 전체 지침을 다시 배치하여 모델의 주의를 환기합니다.
        temp_history = history[:-1] + [{"role": "system", "content": SYSTEM_PROMPT}] + [history[-1]]
    else:
        temp_history = history

    # 템플릿 적용
    prompt = tokenizer.apply_chat_template(
        temp_history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 추론 수행 (Hard Constraint 필터 제외)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.5,
            top_p=0.8,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    
    # 답변 디코딩 및 예외 처리
    if len(generated_tokens) == 0:
        reply = "어르신, 잠시 목소리가 잘 안 들렸어요. 다시 말씀해 주시겠어요?"
    else:
        reply = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        if not reply:
            reply = "어르신, 제가 방금 답변을 준비하다가 깜빡했네요. 다시 대화 나눠요."

    history.append({"role": "assistant", "content": reply})
    return reply

## 5) CSV 로그 저장

- session_id 단위로 대화를 기록합니다.
- 수업에서는 "저장되는 구조를 이해"하는 것이 목표입니다.

In [5]:
CSV_PATH = "chat_log.csv"

def save_log(session_id, user, assistant):
    # 1. 파일 존재 여부 확인
    is_new = not os.path.exists(CSV_PATH)

    try:
        # 2. 로그 기록 (UTF-8 인코딩으로 한글 깨짐 방지)
        with open(CSV_PATH, "a", newline="", encoding="utf-8-sig") as f: # Excel 호환성을 위해 utf-8-sig 사용
            writer = csv.writer(f)
            
            # 첫 생성 시 헤더 추가
            if is_new:
                writer.writerow(["session_id", "timestamp", "user", "assistant"])

            # 대화 데이터 기록
            writer.writerow([
                session_id,
                datetime.now().isoformat(timespec="seconds"),
                user,
                assistant
            ])
    except Exception as e:
        # 임베디드 환경처럼 파일 쓰기 권한이나 용량 문제가 생길 경우를 대비한 간단한 예외 처리
        print(f"로그 저장 중 오류 발생: {e}")

## 6) 위젯 UI(SimpleChatUI)

구성:
- Output: 대화 출력 영역
- Text: 사용자 입력
- Button: 보내기

Enter 전송:
- `Text.on_submit`는 deprecated
- `continuous_update=False + observe(value)`로 Enter/확정 이벤트를 처리합니다.

중복 전송 방지:
- observe 이벤트가 여러 번 들어오는 경우를 막기 위해 `_last_sent_value`를 사용합니다.

In [6]:
import requests
import os
from IPython.display import Audio, display

# 서비스 URL 설정
ASR_URL = "http://127.0.0.1:8009/transcribe"
TTS_URL = "http://127.0.0.1:8010/tts"

# [수정] SYSTEM_PROMPT 정의는 3번 셀에서 통합 관리하므로 여기서는 삭제합니다.

def call_asr(audio_path: str, language="korean", task="transcribe"):
    """음성 파일을 텍스트로 변환 (STT)"""
    with open(audio_path, "rb") as f:
        files = {"audio": (os.path.basename(audio_path), f, "audio/wav")}
        data = {"language": language, "task": task}
        r = requests.post(ASR_URL, files=files, data=data, timeout=600)
    r.raise_for_status()
    return r.json()

def call_tts(text: str, temperature=0.1, top_p=0.6, top_k=50):
    """텍스트를 음성 파일로 변환 (TTS)"""
    data = {
        "text": text,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
    }
    r = requests.post(TTS_URL, data=data, timeout=600)
    r.raise_for_status()
    return r.json()

print("ASR/TTS 연동 함수 로드 완료. (시스템 프롬프트는 3번 셀의 설정을 사용합니다.)")

ASR/TTS 연동 함수 로드 완료. (시스템 프롬프트는 3번 셀의 설정을 사용합니다.)


In [7]:
#사용자의 입력이 '긍정/동의'인지 '부정/거절'인지 LLM이 판단하는 함수

def is_positive_response(user_text):
    """
    사용자의 입력이 '긍정/동의'인지 '부정/거절'인지 LLM이 판단하는 함수
    True: 긍정, False: 부정
    """
    # LLM에게 내리는 '판사' 역할의 지시문
    prompt = f"""
    당신은 문장의 의도를 파악하는 분석기입니다.
    사용자의 답변이 제안에 대해 '동의, 수락, 긍정, 듣고싶음'을 의미하면 'Y',
    '거절, 부정, 나중에, 싫어'를 의미하면 'N'을 출력하세요.
    다른 말은 절대 하지 말고 오직 알파벳 한 글자(Y 또는 N)만 출력하세요.

    질문: 가족 메시지를 들으시겠습니까?
    답변: {user_text}
    판단:"""

    # 텍스트화 (토크나이저 템플릿 없이 raw prompt 사용)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=2,      # 딱 한두 글자만 생성하면 되므로 매우 빠름
            temperature=0.1,       # 창의성 0% (정확한 판단을 위해)
            pad_token_id=tokenizer.eos_token_id
        )
    
    # 결과 디코딩
    result = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().upper()
    
    # 디버깅용 출력 (나중에 주석 처리 가능)
    print(f"[의도파악] 입력: '{user_text}' -> 판단: {result}")
    
    return "Y" in result # Y가 포함되어 있으면 True 반환

In [8]:
import os, uuid, base64, time, shutil, re, json, csv, threading, traceback, wave, contextlib, random
from datetime import datetime
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript

# 0) 경로 설정
BASE_DIR = Path("./family_message")
ARCHIVE_DIR = Path("./saving_family_message")
CSV_DIR = Path("./csv_summary")
JSONL_DIR = Path("./jsonl_summary")
for p in [BASE_DIR, ARCHIVE_DIR, CSV_DIR, JSONL_DIR]: p.mkdir(parents=True, exist_ok=True)

FAM_ID = "fam_recorder_final"
ELD_ID = "elder_recorder_final"

# 1) 유틸리티 함수
def get_wav_duration(p):
    try:
        with contextlib.closing(wave.open(str(p), 'r')) as f:
            return f.getnframes() / float(f.getframerate())
    except: return 2.0

def clean_for_tts(t):
    if not t: return ""
    t = re.sub(r'[^가-힣a-zA-Z0-9\s\.\?\!]', ' ', str(t))
    t = re.sub(r'\.+', '.', t); t = re.sub(r'\?+', '?', t); t = re.sub(r'\!+', '!', t)
    t = re.sub(r'\s+[\.\?\!]', '', t)
    return re.sub(r'\s+', ' ', t).strip()

# 2) 녹음기 UI 생성 함수
def get_recorder_ui(rec_id, label):
    return f"""
<div id="{rec_id}" style="border:2px solid #ddd; padding:15px; border-radius:10px; background:#fdfdfd; margin:10px 0;">
    <div style="font-weight:bold; margin-bottom:10px;">{label}</div>
    <button id="{rec_id}_start" onclick="window.{rec_id}_startFunc()" style="padding:10px 20px; background:#4CAF50; color:white; border:none; border-radius:5px; cursor:pointer;">🎤 녹음 시작</button>
    <button id="{rec_id}_stop" onclick="window.{rec_id}_stopFunc()" disabled style="padding:10px 20px; background:#ccc; color:white; border:none; border-radius:5px; cursor:pointer;">⏹ 녹음 끝</button>
    <span id="{rec_id}_status" style="margin-left:10px; color:#666;">대기 중</span>
</div>
<script>
(function() {{
    let mr; let ch = [];
    window.{rec_id}_startFunc = async () => {{
        const st = document.getElementById("{rec_id}_status");
        try {{
            const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
            mr = new MediaRecorder(stream); ch = [];
            mr.ondataavailable = e => ch.push(e.data);
            mr.onstop = () => {{
                const blob = new Blob(ch, {{ type: 'audio/webm' }});
                const reader = new FileReader();
                reader.readAsDataURL(blob);
                reader.onloadend = () => {{
                    const b64 = reader.result.split(',')[1];
                    const ta = document.querySelector('.target-{rec_id} textarea');
                    if(ta) {{ 
                        ta.value = b64; 
                        ta.dispatchEvent(new Event('input', {{ bubbles: true }})); 
                    }}
                    st.innerText = "전송 완료 ✅";
                }};
            }};
            mr.start(); st.innerText = "🔴 녹음 중..."; 
            document.getElementById("{rec_id}_start").disabled = true;
            document.getElementById("{rec_id}_stop").disabled = false;
        }} catch (e) {{ alert("마이크 권한을 허용해주세요!"); }}
    }};
    window.{rec_id}_stopFunc = () => {{
        if(mr) mr.stop();
        document.getElementById("{rec_id}_start").disabled = false;
        document.getElementById("{rec_id}_stop").disabled = true;
    }};
}})();
</script>
"""

# 3) 통합 클래스
class FinalIntegratedUI:
    def __init__(self):
        self.is_active = False
        self._busy = False
        self.history = [{"role": "system", "content": SYSTEM_PROMPT}]
        self.session_id = uuid.uuid4().hex[:8]
        self.safe_suffix = " (답변 시 이모티콘, 특수기호(~, ^^)를 사용하지 말고 마침표와 물음표만 사용하세요.)"
        
        # --- 반대말 퀴즈 데이터셋 분할 (25개씩 4그룹 = 100쌍) ---
        g1 = [("덥다", "춥다"), ("크다", "작다"), ("길다", "짧다"), ("높다", "낮다"), ("밝다", "어둡다"), ("무겁다", "가볍다"), ("빠르다", "느리다"), ("멀다", "가깝다"), ("많다", "적다"), ("넓다", "좁다"), ("뜨겁다", "차갑다"), ("깨끗하다", "더럽다"), ("기쁘다", "슬프다"), ("강하다", "약하다"), ("부드럽다", "딱딱하다"), ("두껍다", "얇다"), ("깊다", "얕다"), ("어렵다", "쉽다"), ("새롭다", "오래되다"), ("위", "아래"), ("앞", "뒤"), ("안", "밖"), ("오른쪽", "왼쪽"), ("낮", "밤"), ("하늘", "땅")]
        g2 = [("여자", "남자"), ("어른", "아이"), ("부자", "가난함"), ("성공", "실패"), ("시작", "끝"), ("삶", "죽음"), ("사랑", "미움"), ("기쁨", "슬픔"), ("승리", "패배"), ("질문", "대답"), ("판매", "구매"), ("수출", "수입"), ("합격", "불합격"), ("장점", "단점"), ("행복", "불행"), ("과거", "미래"), ("어제", "내일"), ("오전", "오후"), ("입구", "출구"), ("정답", "오답"), ("공격", "방어"), ("전진", "후퇴"), ("상승", "하강"), ("증가", "감소"), ("팽창", "수축")]
        
        self.quiz_data = g1 + g2
        self.current_session_quizzes = random.sample(self.quiz_data, 3)
        # ------------------------------------------------------

        self.fam_html, self.elder_html = widgets.HTML(), widgets.HTML()
        self.audio_html = widgets.HTML(layout=widgets.Layout(display='none'))
        self.fam_b64 = widgets.Textarea(layout=widgets.Layout(width='0px', height='0px', opacity='0'))
        self.fam_b64.add_class(f"target-{FAM_ID}")
        self.elder_b64 = widgets.Textarea(layout=widgets.Layout(width='0px', height='0px', opacity='0'))
        self.elder_b64.add_class(f"target-{ELD_ID}")
        
        self.btn_start = widgets.Button(description="상담 시작", button_style='success', icon='play')
        self.btn_end = widgets.Button(description="상담 종료", button_style='danger', icon='stop', disabled=True)
        
        self.btn_start.on_click(self._start_session)
        self.btn_end.on_click(self._end_session)
        self.fam_b64.observe(self._on_fam_input, names='value')
        self.elder_b64.observe(self._on_elder_input, names='value')
        
        self.fam_msgs = ["[시스템] 가족분들은 상담 시작 전 메시지를 남겨주세요."]
        self.elder_msgs = ["[시스템] '상담 시작'을 누르면 AI와 대화가 시작됩니다."]
        self._refresh_view()

    def _refresh_view(self):
        def wrap(msgs, bid, h):
            cont = "".join([f"<div style='margin-bottom:5px;'>{m}</div>" for m in msgs])
            js = f"var d=document.getElementById('{bid}'); if(d) d.scrollTop=d.scrollHeight;"
            return f'<div id="{bid}" style="height:{h}; overflow-y:auto; border:1px solid #ccc; padding:10px; background:white;">{cont}<img src="x" style="display:none;" onerror="{js}"></div>'
        self.fam_html.value = wrap(self.fam_msgs, "fam_win", "150px")
        self.elder_html.value = wrap(self.elder_msgs, "eld_win", "350px")

    def _start_session(self, b):
        self.is_active = True
        self.btn_start.disabled, self.btn_end.disabled = True, False
        fam_cnt = len(list(BASE_DIR.glob("*.wav")))
        if fam_cnt > 0:
            self.mode, self.step = "check_msg", 0
            msg = f"어르신, 안녕하세요! 가족 메시지가 {fam_cnt}통 와 있어요. 지금 바로 들려드릴까요?"
        else:
            self.mode, self.step = "step2_greeting", 1
            instruction = "[2단계 1턴 시작] 어르신께 인사를 건네고 수면 안부를 물어보세요." + self.safe_suffix
            msg = chat_llm(self.history, instruction)
        self._add_chat("AI", msg); self.history.append({"role": "assistant", "content": msg})
        time.sleep(self._speak(msg))

    def _add_chat(self, role, text):
        label = "<b>[AI 상담사]</b>" if role == "AI" else "<b>[어르신]</b>"
        self.elder_msgs.append(f"{label} {text}"); self._refresh_view()
        save_log(self.session_id, "User" if role=="User" else "Assistant", text) 

    def _speak(self, text):
        clean = clean_for_tts(text)
        if not clean: return 2.0
        res = call_tts(clean)
        if res.get('wav_path'):
            duration = get_wav_duration(res['wav_path'])
            b64 = base64.b64encode(Path(res['wav_path']).read_bytes()).decode("ascii")
            self.audio_html.value = f'<audio autoplay><source src="data:audio/wav;base64,{b64}" type="audio/wav"></audio>'
            return duration
        return 2.0

    def _on_fam_input(self, change):
        if not change['new']: return
        unique_id = f"{int(time.time())}_{uuid.uuid4().hex[:4]}"
        fname = f"msg_{unique_id}.wav"
        audio_bytes = base64.b64decode(change['new'])
        (BASE_DIR / fname).write_bytes(audio_bytes); (ARCHIVE_DIR / fname).write_bytes(audio_bytes)
        self.fam_msgs.append(f"✅ 메시지 저장 완료! (현재 총 {len(list(BASE_DIR.glob('*.wav')))}통)"); self._refresh_view()
        self.fam_b64.value = ""

    def _on_elder_input(self, change):
        if not self.is_active or not change['new'] or self._busy: return
        self._busy = True
        display(Javascript(f'document.getElementById("{ELD_ID}_start").disabled = true;'))
        threading.Thread(target=self._proc_logic, args=(change['new'],)).start()
        self.elder_b64.value = ""

    def _proc_logic(self, b64):
        try:
            tmp_p = f"/tmp/e_{uuid.uuid4().hex[:4]}.wav"
            Path(tmp_p).write_bytes(base64.b64decode(b64))
            utext = call_asr(tmp_p).get('text', '').strip()
            if not utext or not re.sub(r'[^가-힣a-zA-Z0-9]', '', utext): return
            self._add_chat("User", utext); self.history.append({"role": "user", "content": utext})
            
            instruction = ""
            if self.mode == "check_msg":
                if is_positive_response(utext):
                    for i, f in enumerate(sorted(list(BASE_DIR.glob("*.wav"))), 1):
                        intro = f"{i}번째 메시지입니다."
                        self.elder_msgs.append(f"<b>[AI]</b> {intro}"); self._refresh_view()
                        time.sleep(self._speak(intro) + 0.5); time.sleep(self._play_audio_direct(f) + 1.0)
                    self.mode, self.step = "step2_greeting", 1
                    instruction = "메시지 전달 완료. [2단계 1턴] 공감 후 수면 안부를 물어보세요."
                else:
                    self.mode, self.step = "step2_greeting", 1
                    instruction = "거절 확인. [2단계 1턴] 바로 인사를 건네고 밤새 잘 주무셨는지 물으세요."

            elif self.mode == "step2_greeting":
                if self.step == 1: self.step = 2; instruction = "[2단계 2턴] 공감 후 신체 아픈 곳은 없는지 물으세요."
                else: self.mode, self.step = "step3_reminiscence", 1; instruction = "[3단계 1턴] 공감 후 과거 회상 질문을 하세요."

            elif self.mode == "step3_reminiscence":
                if self.step < 3: self.step += 1; instruction = f"회상 [{self.step}턴]. 공감하고 대화를 이어가세요."
                else:
                    self.mode, self.step = "step4_quiz", 1
                    q1 = self.current_session_quizzes[0][0]
                    instruction = f"회상 완료. [4단계 1턴] 반대말 게임을 제안하며 첫 문제로 '{q1}'를 내세요."

            elif self.mode == "step4_quiz":
                # 현재 어르신의 답변은 (self.step - 1) 인덱스 문제에 대한 답변임
                q_idx = self.step - 1
                prev_q, prev_a = self.current_session_quizzes[q_idx]
                
                # 정답 판별
                correct = prev_a in utext.replace(" ", "")
                judge = f"어르신의 답변은 '{utext}'였고, '{prev_q}'의 실제 정답은 '{prev_a}'입니다. "
                judge += "잘 맞히셨다고 칭찬해 주세요." if correct else "아쉽지만 정답을 친절히 알려주세요."
                
                if self.step < 5:
                    # 다음 문제 제시
                    next_q = self.current_session_quizzes[self.step][0]
                    instruction = f"[4단계 {self.step+1}턴] {judge} 이어서 다음 문제로 '{next_q}'를 내세요."
                    self.step += 1
                else:
                    # 퀴즈 종료
                    instruction = f"[4단계 결과 처리] {judge} 오늘 퀴즈를 모두 마쳤으니 따뜻한 작별 인사를 건네세요."
                    self.mode = "finished"

            # 답변 생성 및 재생
            if self.mode == "finished":
                reply = chat_llm(self.history, instruction + self.safe_suffix)
                self._finalize(reply); return

            reply = chat_llm(self.history, instruction + self.safe_suffix)
            self._add_chat("AI", reply); self.history.append({"role": "assistant", "content": reply})
            time.sleep(self._speak(reply))
            
        finally:
            self._busy = False
            display(Javascript(f'document.getElementById("{ELD_ID}_start").disabled = false;'))
            display(Javascript(f'document.getElementById("{ELD_ID}_status").innerText = "대기 중";'))

    def _play_audio_direct(self, p):
        duration = get_wav_duration(p)
        b64 = base64.b64encode(p.read_bytes()).decode("ascii")
        self.audio_html.value = f'<audio autoplay><source src="data:audio/wav;base64,{b64}" type="audio/wav"></audio>'
        return duration

    def _finalize(self, msg):
        self._add_chat("AI", msg); time.sleep(self._speak(msg)); self._end_session(None)

    def _end_session(self, b):
        self.is_active = False
        self.btn_start.disabled, self.btn_end.disabled = False, True
        self.elder_msgs.append("[시스템] 상담 종료."); self._refresh_view()
        [f.unlink() for f in BASE_DIR.glob("*.wav")]

    def render(self):
        display(HTML(get_recorder_ui(FAM_ID, "👨‍👩‍👧‍👦 가족용 메시지함")), self.fam_html, self.fam_b64, HTML("<hr>"))
        display(widgets.HBox([self.btn_start, self.btn_end]), self.elder_html, self.audio_html)
        display(HTML(get_recorder_ui(ELD_ID, "👵 어르신용 상담창")), self.elder_b64)

ui = FinalIntegratedUI()
ui.render()

HTML(value='<div id="fam_win" style="height:150px; overflow-y:auto; border:1px solid #ccc; padding:10px; backg…

Textarea(value='', layout=Layout(height='0px', width='0px'), _dom_classes=('target-fam_recorder_final',))

HTML(value='<div id="eld_win" style="height:350px; overflow-y:auto; border:1px solid #ccc; padding:10px; backg…

HTML(value='', layout=Layout(display='none'))

Textarea(value='', layout=Layout(height='0px', width='0px'), _dom_classes=('target-elder_recorder_final',))